# tSCS EMG — the four comparison figures (subject NTA, 24-07-2026)

One session, one participant: **stimulation mode** (30 Hz burst / ARC-EX) × **polarity**
(cathodic / anodic) × **lidocaine** (before / with). Four figures, each holding one thing fixed:

| figure | fixed | compared | with |
|---|---|---|---|
| **1** | anodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **2** | cathodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **3** | ARC-EX | anodic vs cathodic | before and with lidocaine |
| **4** | 30 Hz burst | anodic vs cathodic | before and with lidocaine |

## Colours and styles
**gray = before lidocaine, orange = with lidocaine** in every figure. The second factor is the
**style**: in figures 1–2 **solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX**; in
figures 3–4 **solid / plain = cathodic, dashed / hatched = anodic**.

**Intensities:** `BURST_MA` and `ARCEX_MA` in the config set the mA of 30 Hz and of ARC-EX in
every figure; the `AMP[...]` lines below them override any single train (polarity × lidocaine)
when you want each at its own motor threshold — the log gives burst cathodic 25 mA before / 30
with lidocaine, anodic 30 / 30; ARC-EX cathodic 70 / 70, anodic 65 / 90. `FIGS[n]` sets the muscle
of each figure's one-muscle version (and can override its intensities too).

Every figure also comes with **waterfalls** — all intensities of each condition stacked, with the
sweep the analysis uses drawn in **orange**, so the whole recruitment is visible at a glance.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, pretty, waterfall, detect_pulses
from functions.burst import (compare_at_intensity, summary_curves, plot_pulse_overlay,
                             burst_p2p, detection_report, resolve_muscles, artifact_extent)
from functions.paper import (fig_train_modes, fig_depression, depression_stats,
                             motor_thresholds, muscles_with_threshold, fig_thresholds,
                             step_up)
set_style()


## Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
FILE = {   # (mode, polarity, lidocaine state) -> file
    ("burst", "cathodic", "before"):    "Burst_autosave_20260724_102443_670ms.csv",
    ("burst", "cathodic", "lidocaine"): "Burst_autosave_20260724_113834_522ms.csv",
    ("burst", "anodic",   "before"):    "Burst_autosave_20260724_102721_694ms.csv",
    ("burst", "anodic",   "lidocaine"): "Burst_autosave_20260724_114055_891ms.csv",
    ("arcex", "cathodic", "before"):    "Modulated_autosave_20260724_103530_508ms.csv",
    ("arcex", "cathodic", "lidocaine"): "Modulated_autosave_20260724_114332_473ms.csv",
    ("arcex", "anodic",   "before"):    "Modulated_autosave_20260724_103849_459ms.csv",
    ("arcex", "anodic",   "lidocaine"): "Modulated_autosave_20260724_114730_561ms.csv",
}
NAME = {"burst": "30 Hz", "arcex": "ARC-EX", "cathodic": "cathodic", "anodic": "anodic"}
LIDO_COL = {"before": "0.45", "lidocaine": "#f39c12"}       # colour = lidocaine
STYLE_A  = ("",    "-")                                      # first compared value: plain / solid
STYLE_B  = ("///", "--")                                     # second: hatched / dashed

# ---- the intensity of every train, in one table -------------------------------------------
BURST_MA, ARCEX_MA = 35, 110      # <-- the mA used for 30 Hz and for ARC-EX, everywhere
AMP = {k: (BURST_MA if k[0] == "burst" else ARCEX_MA) for k in FILE}

# per-condition overrides - uncomment/edit any line to give that train its own mA
AMP[("burst", "cathodic", "before")]    = 35      # log motor threshold: 25
AMP[("burst", "cathodic", "lidocaine")] = 35
AMP[("burst", "anodic",   "before")]    = 35
AMP[("burst", "anodic",   "lidocaine")] = 35
AMP[("arcex", "cathodic", "before")]    = 75
AMP[("arcex", "cathodic", "lidocaine")] = 75
AMP[("arcex", "anodic",   "before")]    = 105
AMP[("arcex", "anodic",   "lidocaine")] = 105

# Muscles that keep the hand-picked mA below instead of their own motor threshold. Everything
# not listed here is plotted at ITS OWN threshold when the figure has mt=True.
MT_PIN = ["Flex. digitorum (R)"]

FIG_SETTINGS = {   # <-- everything each figure plots: the mA of each compared condition,
                   #     the muscle of its one-muscle version, and its waterfall muscles
                   #     mt=True -> every muscle at its own motor threshold, except MT_PIN,
                   #     which stays at the `amps` below. mt=False -> that mA for all muscles.
    1: dict(amps={"burst": BURST_MA, "arcex": ARCEX_MA}, mt=True,
            muscle="Flex. digitorum (R)", wf=None),   # fig 1: anodic · 30 Hz vs ARC-EX
    2: dict(amps={"burst": BURST_MA, "arcex": ARCEX_MA}, mt=True,
            muscle="Flex. digitorum (R)", wf=None),   # fig 2: cathodic · 30 Hz vs ARC-EX
    3: dict(amps={"cathodic": 105, "anodic": 80}, mt=True,
            muscle="Flex. digitorum (R)", wf=None),   # fig 3: ARC-EX · cathodic vs anodic
    4: dict(amps={"cathodic": BURST_MA, "anodic": BURST_MA}, mt=True,
            muscle="Flex. digitorum (R)", wf=None),   # fig 4: 30 Hz burst · cathodic vs anodic
}

FIGS = {   # what each figure holds fixed and what it compares (fixed by the design, not settings)
    1: dict(fixed=("polarity", "anodic"),   compare=("burst", "arcex")),
    2: dict(fixed=("polarity", "cathodic"), compare=("burst", "arcex")),
    3: dict(fixed=("mode", "arcex"),        compare=("cathodic", "anodic")),
    4: dict(fixed=("mode", "burst"),        compare=("cathodic", "anodic")),
}
XLIM_WF   = (-20, 130)      # time window of the waterfalls (ms)
WF_MUSCLES = None           # None = all muscles in the waterfalls, or a list, e.g. ["Flex. digitorum (R)"]

# ---- peak-detection parameters (see the section "How the peaks are detected" below) --------
N_PULSES      = 10      # pulses of the train analysed
RESP_START_MS = 8.0     # response window starts this long after EACH pulse onset - must clear the
                        # artifact. A dict overrides single channels: {"R_DELmed": 11.0}
RESP_END_MS   = None    # window end, also from the pulse onset; None = run to the next pulse
GUARD_MS      = 1.0     # ...minus this
SNR_ON        = "median" # which pulses the criterion looks at: "median" (the train's median
                        # pulse), "p1" (pulse 1 only), "half" (at least half the pulses).
                        # "p1" throws away a whole train when its first pulse happens to be small.
MIN_SNR       = 1.2     # keep a train only if pulse-1 p2p >= MIN_SNR x baseline p2p (None = keep all)
ANCHOR, ANCHOR_WIN_MS = "mean", 3.0   # anchor each pulse's max/min to the train average
MAX_EDGE_FRAC = 0.5     # reject a train as artifact if more than this fraction of its pulses have
                        # max/min on a window border (None = off)
EDGE_MS       = 1.0     # "on a border" means within this many ms of it
JITTER_MS     = 0.5     # flag a pulse whose peak latency differs from the train median by more than this
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, resp_end_ms=RESP_END_MS,
          guard_ms=GUARD_MS, min_snr=MIN_SNR, max_edge_frac=MAX_EDGE_FRAC, snr_on=SNR_ON,
          anchor=ANCHOR, anchor_win_ms=ANCHOR_WIN_MS)

muscles = [c for c in load_run(D + FILE[("burst", "cathodic", "before")])[2] if c != "Trigger A"]

# ---- which muscles the NUMBERS come from -------------------------------------------------
EXCLUDE = ["Deltoid med.", "Biceps", "Triceps long"]   # proximal muscles: kept in the waterfalls,
                                                       # never used for peak-to-peak
ANALYSE = [c for c in muscles if not any(pretty(c).startswith(x) for x in EXCLUDE)]
print("analysed: " + ", ".join(pretty(c) for c in ANALYSE))
print("waterfalls only: " + ", ".join(pretty(c) for c in muscles if c not in ANALYSE))

def build(n, amps=None, muscle=None, mt=None):
    """(amps / muscle / mt default to FIG_SETTINGS[n] above.)"""
    """Everything figure `n` needs, ordered A-before, A-lidocaine, B-before, B-lidocaine.
    amps: {compared value: mA} for this figure (default: the AMP table above).
    mt:   True  -> each muscle at ITS OWN motor threshold (MT, from the cell above), except the
                   muscles in MT_PIN, which stay at `amps`. Muscles with no threshold in one of
                   the compared conditions are left out of the figure.
          False -> one mA for every muscle, the old behaviour.
          The threshold is always the one measured BEFORE lidocaine, used for both the before and
          the with-lidocaine train, so the only thing that differs between them is the lidocaine.
    muscle: the muscle of the one-muscle version and of the focused waterfalls."""
    spec = FIGS[n]; fixed, compare = spec["fixed"], spec["compare"]
    amps = amps if amps is not None else FIG_SETTINGS[n]["amps"]
    if amps == "MT":                    # amps="MT" is the same as mt=True
        amps, mt = FIG_SETTINGS[n]["amps"], True
    use_mt = FIG_SETTINGS[n].get("mt", False) if mt is None else mt
    muscle = muscle if muscle is not None else FIG_SETTINGS[n]["muscle"]
    files, labels, colours, hatches, lss, amp_list, keys = [], [], [], [], [], [], []
    for val, (hat, ls) in zip(compare, (STYLE_A, STYLE_B)):
        for state in ("before", "lidocaine"):
            key = (val, fixed[1], state) if fixed[0] == "polarity" else (fixed[1], val, state)
            keys.append(key); files.append(D + FILE[key])
            labels.append(f"{NAME[key[0]]} · {key[1]} · {'before lido' if state == 'before' else 'with lido'}")
            colours.append(LIDO_COL[state]); hatches.append(hat); lss.append(ls)
            pin = amps.get(val, AMP[key])
            if use_mt:
                amp_list.append(mt_amp((key[0], key[1], "before"), pin=pin))
            else:
                amp_list.append(pin)
    return dict(files=files, labels=labels, colours=colours, hatches=hatches, linestyles=lss,
                amps=tuple(amp_list), keys=keys, muscle=muscle, n=n)

def peaks(cfg, muscle=None, report=True, kw=None):
    """Look at HOW the peaks are detected for this figure, at the mA it uses.
    One pulse-overlay per condition: every pulse re-aligned to its own onset, green = the response
    window, v / ^ = the max / min taken, red ring = flagged (on a window border, or at a different
    latency than the other pulses). Stacked markers = the same deflection every time.
    muscle: one label / list (default: the figure's muscle) · kw: override the detection parameters."""
    kw = kw or KW
    ms = muscle or cfg["muscle"] or ANALYSE
    for key, f_, a in zip(cfg["keys"], cfg["files"], cfg["amps"]):
        meta_, t_, sig_ = load_run(f_)
        lab = f"{NAME[key[0]]} · {key[1]} · {'before lido' if key[2] == 'before' else 'with lido'}"
        chans = resolve_muscles([c for c in sig_ if c != "Trigger A"],
                                ms if isinstance(ms, list) else [ms])
        by_ma = {}                       # muscles grouped by the mA they are plotted at
        for m_ in chans:
            v = amp_of(a, m_)
            if v is not None:
                by_ma.setdefault(v, []).append(m_)
        for v, grp in sorted(by_ma.items()):
            plot_pulse_overlay(meta_, t_, sig_, grp, amp=v, edge_ms=EDGE_MS, jitter_ms=JITTER_MS,
                               title=f"{lab} — {v} mA", **kw)
            if report:
                res = burst_p2p(meta_, t_, sig_, grp, **kw)
                print(f"----- {lab} — {v} mA")
                detection_report(res, grp, edge_ms=EDGE_MS, jitter_ms=JITTER_MS)
                print(f"  artifact spike width (ms after onset): "
                      + ", ".join(f"{pretty(m)} {x:.1f}" for m, x in
                                  artifact_extent(t_, sig_, grp, detect_pulses(t_, sig_["Trigger A"])[:N_PULSES],
                                                  w=[x["amp_ma"] for x in meta_].index(v)).items()))

def amp_of(a_, m_):
    """The mA one muscle is plotted at: a number, or its entry in a {muscle: mA} dict."""
    if not isinstance(a_, dict):
        return a_
    for k in (m_, pretty(m_)):
        if k in a_:
            return a_[k]
    return a_.get("default")

def check(cfg):
    """Print what each train of this figure will be plotted at, and whether that mA exists."""
    for key, f_, a in zip(cfg["keys"], cfg["files"], cfg["amps"]):
        avail = sorted({m["amp_ma"] for m in load_run(f_)[0]})
        head = f"{NAME[key[0]]:7s} {key[1]:9s} {key[2]:10s}"
        if not isinstance(a, dict):
            print(f"{head} -> {a:4d} mA, every muscle "
                  + ("ok" if a in avail else f"MISSING - available: {avail}"))
            continue
        ok = {m: v for m, v in sorted(a.items()) if v in avail}
        bad = {m: v for m, v in sorted(a.items()) if v not in avail}
        print(f"{head} -> own motor threshold, {len(ok)} muscle(s): "
              + ", ".join(f"{m} {v}{' (pinned)' if m in MT_PIN else ''}" for m, v in ok.items()))
        if bad:
            print(f"{'':29s} MISSING from this file: "
                  + ", ".join(f"{m} {v}" for m, v in bad.items()) + f" - available: {avail}")
        gone = [pretty(m) for m in ANALYSE if amp_of(a, m) is None]
        if gone:
            print(f"{'':29s} no threshold, left out: " + ", ".join(gone))

def show(cfg, title, muscle=None):
    """The comparison figure; muscle=cfg["muscle"] for the one-muscle version."""
    compare_at_intensity(cfg["files"], None, amp=cfg["amps"], normalize="none",
                         muscles=muscle or ANALYSE,
                         labels=cfg["labels"], colours=cfg["colours"], hatches=cfg["hatches"],
                         linestyles=cfg["linestyles"], title=title, **KW)

def wf(cfg, prefix, muscles_=None):
    """One waterfall per condition: every intensity stacked, and the sweep the analysis uses
    drawn in ORANGE. All four share the gain of the first, so the heights are comparable."""
    ms = resolve_muscles(muscles, muscles_ or WF_MUSCLES)      # channel names or labels
    gains = None
    for lab, f_, a in zip(cfg["labels"], cfg["files"], cfg["amps"]):
        g = waterfall(*load_run(f_), ms, xlim=XLIM_WF, gains=gains, highlight=a,
                      highlight_label=f"analysed · {AT}", title=f"{prefix} · {lab}")
        gains = gains or g

for k, f_ in FILE.items():                        # what each recording contains
    print(f"{NAME[k[0]]:7s} {k[1]:9s} {k[2]:10s} {[m['amp_ma'] for m in load_run(D + f_)[0]]} mA")


## How the peaks are detected — and how to check it

For every pulse the peak-to-peak is `max − min` inside a window that starts **after that pulse's
artifact**:

    window_k = [ pulse_k onset + RESP_START_MS , pulse_(k+1) onset − GUARD_MS ]   (or + RESP_END_MS)

Then two filters: a train counts as a response only if **pulse-1 p2p ≥ `MIN_SNR` × the
pre-stimulus baseline p2p**, and a train is **rejected as artifact** if more than `MAX_EDGE_FRAC`
of its pulses have their max/min sitting within `EDGE_MS` of a window border (a smooth
artifact-recovery curve has no peak inside the window, so its extremes fall on the edges).

| parameter | raise it when | lower it when |
|---|---|---|
| `RESP_START_MS` | the ▼/▲ land on the artifact decay right after the pulse | the window cuts off the start of a real response |
| `RESP_END_MS` | — | the window catches something late that isn't the response |
| `MIN_SNR` | noise-only trains are being kept | real small responses are dropped |
| `MAX_EDGE_FRAC` | a real response is being called artifact | artifact recovery is being kept |
| `ANCHOR` | the detector jumps between two candidate peaks | you want a free search |
| `JITTER_MS` | too many pulses flagged on a variable-latency muscle | you want stricter flagging |

`peaks(FIGn)` below draws, for each condition **at the mA that figure uses**, every pulse
re-aligned to its own onset: **green = the response window, ▼/▲ = the max/min actually taken, red
ring = flagged**. If the detection is right, the markers stack on top of each other. It also
prints the flag counts and the measured artifact width per channel, so you can set
`RESP_START_MS` above it.

To try other settings without touching the config, pass `kw=`:
```python
peaks(FIG1, kw=dict(KW, resp_start_ms=11.0, min_snr=3.0))
```

**Which max and min each pulse gets (`ANCHOR`)** — when a waveform has two candidate peaks (say a
dip at 11 ms and another at 20 ms around a peak at 15 ms), a plain `argmax`/`argmin` picks
whichever happens to be larger on that pulse, so it jumps between them and the pulses get flagged
for "jitter" even though the response is perfectly consistent. With `ANCHOR`, the latency of the
max and of the min is taken **from the train itself** — the average of its pulses (`"mean"`), or
the first one / first two (`"first"`, `"first2"`) — and each pulse is then searched only within
`ANCHOR_WIN_MS` of those latencies. Every pulse is measured on the *same* deflection.
`ANCHOR = None` restores the independent per-pulse search.

**A floor on `JITTER_MS`.** Latencies can only differ in whole samples (here **0.79 ms**), so a
tolerance below that flags a pulse that is one sample off the median — which is why 0.5 ms flags
almost everything. And with `ANCHOR` on, every pulse is already searched within `ANCHOR_WIN_MS` of
the same reference, so a tighter tolerance cannot mean anything. The flag rule therefore applies
`max(JITTER_MS, 1.5 × sampling interval, ANCHOR_WIN_MS)` and prints the value it used.

**Which pulses the criterion looks at (`SNR_ON`).** Testing **pulse 1 only** rejects a whole train
whenever its first pulse happens to be small — e.g. ARC-EX cathodic 110 mA on Flex. digitorum (R),
where pulse 1 sits at SNR 1.16 while the other nine are at 1.9–3.8 and the response is obvious.
`SNR_ON = "median"` (default) uses the train's median pulse, `"half"` asks that at least half the
pulses clear the threshold, `"p1"` is the old behaviour. When a train is kept but its **first**
pulse is near noise, the diagnostics print a warning — the *% of pulse 1* numbers are unreliable
for that train, even though its raw mV values are fine.


## Motor threshold per muscle — the intensity used for the comparisons

Every muscle is recruited at a different intensity, so comparing all of them at one fixed mA mixes
muscles that are far above threshold with muscles that are still below it. Here each muscle gets
**its own motor threshold**: the lowest intensity at which its response appears — pulse-1
peak-to-peak above `MIN_SNR` × the pre-stimulus baseline, not artifact-rejected, and **still
passing at the next intensity** (`CONSECUTIVE = 2`), so a single noisy intensity cannot be
mistaken for a threshold.

Three steps: **detect** → **look at the recruitment curves and correct what is wrong** → **use it**.
The figure shows, per muscle, pulse-1 p2p against intensity: filled markers = passes the
criterion, hollow = does not, dashed line = the threshold that will be used.

In [ ]:

# ======== settings ========
CONSECUTIVE = 2            # intensities the response criterion must hold for
STATE       = "before"     # the lidocaine state every comparison below is run at
# ==========================

MT_STEPS    = 1            # analyse this many intensity steps ABOVE each muscle's threshold.
                           # 0 = at the threshold itself (the smallest response the criterion
                           # accepts); 1 = the next intensity that recording contains.
MT_MISSING  = "drop"       # when the threshold is already the highest intensity tested:
                           # "drop" = leave that muscle out, "clip" = keep it at the top
AT = "MT" + (f"+{MT_STEPS}" if MT_STEPS else "")    # how the figures name that intensity
MT_COL = {"cathodic": "#1f3b73", "anodic": "#e6550d"}

def mt_specs(mode, state):
    return [dict(label=pol, csv=D + FILE[(mode, pol, state)], colour=MT_COL[pol],
                 hatch=("" if pol == "cathodic" else "///")) for pol in ("cathodic", "anodic")]

MT = {}                    # MT[(mode, polarity, state)] = {muscle: mA}
for mode in ("burst", "arcex"):
    for state in ("before", "lidocaine"):
        got = motor_thresholds(mt_specs(mode, state), ANALYSE, consecutive=CONSECUTIVE,
                               verbose=False, **KW)
        for pol in ("cathodic", "anodic"):
            MT[(mode, pol, state)] = got[pol]

def mt_amp(key, pin=None):
    """{muscle: mA} to ANALYSE (mode, polarity, state) at: each muscle's own motor threshold
    moved MT_STEPS steps up the ladder of that recording. The muscles in MT_PIN keep the
    hand-picked intensity `pin` (a number or a {muscle: mA} dict; default: the AMP table)."""
    per = ({m: v for m, v in MT[key].items() if v} if not MT_STEPS else
           step_up(MT[key], D + FILE[key], MT_STEPS, MT_MISSING, verbose=False))
    pin = AMP[key] if pin is None else pin
    for m in MT_PIN:
        v = pin.get(m) if isinstance(pin, dict) else pin
        if v:
            per[m] = v
    return per

print("motor threshold per muscle (mA) - first intensity with a response, held for "
      f"{CONSECUTIVE} steps\n")
names = sorted({m for d in MT.values() for m in d})
print(f"{'muscle':22s}" + "".join(f"{f'{m[:5]} {p[:4]} {s[:3]}':>16s}" for m, p, s in MT))
for n_ in names:
    print(f"{n_:22s}" + "".join(f"{str(MT[k].get(n_) or '-'):>16s}" for k in MT))

if MT_STEPS:
    print(f"\nanalysed {MT_STEPS} step(s) above threshold - muscles with no intensity left:")
    for k in MT:
        step_up(MT[k], D + FILE[k], MT_STEPS, MT_MISSING, label=f"{k[0]:6s} {k[1]:9s} {k[2]}")


### Check the thresholds on the recruitment curves

Look for two mistakes: a threshold sitting on a **flat, noisy curve** (the criterion passed on
noise — typically at the lowest intensity tested), and a muscle with an obvious response whose
threshold was **not** found. Correct either in the next cell.

In [ ]:

# the recruitment curve of every muscle, for BOTH protocols: filled = the train passes the
# response criterion, dashed line = the motor threshold taken from it
for mode_ in ("burst", "arcex"):
    fig_thresholds(mt_specs(mode_, STATE), ANALYSE, consecutive=CONSECUTIVE,
                   MT={pol: MT[(mode_, pol, STATE)] for pol in ("cathodic", "anodic")},
                   title=f"{NAME[mode_]} · {STATE} lidocaine — motor threshold per muscle", **KW)


### Correct the thresholds by hand

`MT[polarity][muscle] = mA` sets one, `= None` drops that muscle from the comparison. Re-run the
figure above to check. Only muscles with a threshold in **both** polarities can be compared, and
those are listed by `muscles_with_threshold`.

In [ ]:

# --- manual corrections: MT[(mode, polarity, state)][muscle] = mA, or None to drop the muscle ---
# MT[("burst", "cathodic", "before")]["Triceps long (L)"] = None   # threshold on a flat, noisy curve
# MT[("burst", "anodic",   "before")]["Flex. digitorum (R)"] = 30
# ------------------------------------------------------------------------------------------------

def mt_common(mode, state):
    """Muscles with a threshold in BOTH polarities for that (mode, state) - the comparable set."""
    a, b = mt_amp((mode, "cathodic", state)), mt_amp((mode, "anodic", state))
    return [m for m in sorted(a) if a.get(m) and b.get(m)]

for mode in ("burst", "arcex"):
    for state in ("before", "lidocaine"):
        ms = mt_common(mode, state)
        print(f"{mode:6s} {state:10s} {len(ms):2d} comparable muscles: " + ", ".join(ms))


## Figure 1 · 30 Hz vs ARC-EX — **anodic**, before and with lidocaine

Plain bars / solid lines = **30 Hz burst**, hatched / dashed = **ARC-EX**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG1 = build(1)                 # settings: FIG_SETTINGS[1] in the config cell
# FIG1 = build(1, mt=False)                     # ...or override just here:
# FIG1 = build(1, amps={...}, muscle="Biceps (R)")     # mt=False = one mA for every muscle
check(FIG1)


### Figure 1a · all muscles

In [ ]:
show(FIG1, "Fig 1 · anodic — all muscles")


### Figure 1b · one muscle

In [ ]:
show(FIG1, f"Fig 1 · anodic — {FIG1['muscle']}", muscle=FIG1["muscle"])


### Figure 1c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG1)                      # the figure's muscle; peaks(FIG1, muscle="Biceps (R)") for another
# peaks(FIG1, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 1d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**. All share one
gain per muscle, so heights are comparable. **Orange = the sweep the analysis uses**: each
muscle's own motor threshold, moved `MT_STEPS` intensities up.

In [ ]:
wf(FIG1, "Fig 1", muscles_=FIG_SETTINGS[1]["wf"])


## Figure 2 · 30 Hz vs ARC-EX — **cathodic**, before and with lidocaine

Plain / solid = **30 Hz burst**, hatched / dashed = **ARC-EX**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG2 = build(2)                 # settings: FIG_SETTINGS[2] in the config cell
# FIG2 = build(2, mt=False)                     # ...or override just here:
# FIG2 = build(2, amps={...}, muscle="Biceps (R)")     # mt=False = one mA for every muscle
check(FIG2)


### Figure 2a · all muscles

In [ ]:
show(FIG2, "Fig 2 · cathodic — all muscles")


### Figure 2b · one muscle

In [ ]:
show(FIG2, f"Fig 2 · cathodic — {FIG2['muscle']}", muscle=FIG2["muscle"])


### Figure 2c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG2)                      # the figure's muscle; peaks(FIG2, muscle="Biceps (R)") for another
# peaks(FIG2, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 2d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**. All share one
gain per muscle, so heights are comparable. **Orange = the sweep the analysis uses**: each
muscle's own motor threshold, moved `MT_STEPS` intensities up.

In [ ]:
wf(FIG2, "Fig 2", muscles_=FIG_SETTINGS[2]["wf"])


## Figure 3 · ARC-EX — **cathodic vs anodic**, before and with lidocaine

Plain / solid = **cathodic**, hatched / dashed = **anodic**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG3 = build(3)                 # settings: FIG_SETTINGS[3] in the config cell
# FIG3 = build(3, mt=False)                     # ...or override just here:
# FIG3 = build(3, amps={...}, muscle="Biceps (R)")     # mt=False = one mA for every muscle
check(FIG3)


### Figure 3a · all muscles

In [ ]:
show(FIG3, "Fig 3 · ARC-EX — all muscles")


### Figure 3b · one muscle

In [ ]:
show(FIG3, f"Fig 3 · ARC-EX — {FIG3['muscle']}", muscle=FIG3["muscle"])


### Figure 3c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG3)                      # the figure's muscle; peaks(FIG3, muscle="Biceps (R)") for another
# peaks(FIG3, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 3d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**. All share one
gain per muscle, so heights are comparable. **Orange = the sweep the analysis uses**: each
muscle's own motor threshold, moved `MT_STEPS` intensities up.

In [ ]:
wf(FIG3, "Fig 3", muscles_=FIG_SETTINGS[3]["wf"])


### Figure 3e · across intensities

In [ ]:
summary_curves(FIG3["files"], ANALYSE, labels=FIG3["labels"], colours=FIG3["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Figure 4 · 30 Hz burst — **cathodic vs anodic**, before and with lidocaine

Plain / solid = **cathodic**, hatched / dashed = **anodic**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** — the mA of each compared condition, the muscle, the waterfall
muscles — are in `FIG_SETTINGS` in the config cell at the top.

In [ ]:
FIG4 = build(4)                 # settings: FIG_SETTINGS[4] in the config cell
# FIG4 = build(4, mt=False)                     # ...or override just here:
# FIG4 = build(4, amps={...}, muscle="Biceps (R)")     # mt=False = one mA for every muscle
check(FIG4)


### Figure 4a · all muscles

In [ ]:
show(FIG4, "Fig 4 · 30 Hz burst — all muscles")


### Figure 4b · one muscle

In [ ]:
show(FIG4, f"Fig 4 · 30 Hz burst — {FIG4['muscle']}", muscle=FIG4["muscle"])


### Figure 4c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG4)                      # the figure's muscle; peaks(FIG4, muscle="Biceps (R)") for another
# peaks(FIG4, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 4d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**. All share one
gain per muscle, so heights are comparable. **Orange = the sweep the analysis uses**: each
muscle's own motor threshold, moved `MT_STEPS` intensities up.

In [ ]:
wf(FIG4, "Fig 4", muscles_=FIG_SETTINGS[4]["wf"])


### Figure 4e · across intensities

In [ ]:
summary_curves(FIG4["files"], ANALYSE, labels=FIG4["labels"], colours=FIG4["colours"],
               markers=["o", "o", "s", "s"], **KW);



## Paper-style figure — 30 Hz vs ARC-EX, before and with lidocaine

Four conditions on one figure: **black = 30 Hz burst, red = ARC-EX** (mode) · **solid, plain bars
= before lidocaine; dashed, hatched = with lidocaine**.

**Left** — the 10 pulses of each condition, one row per muscle, with a mV scale bar; the mA used
by each condition for *that* muscle is written in its colour at the top right of the panel.
**Right** — pulse 1 (outlined, = 100 %) vs the mean of pulses 2–10 (filled), each condition
relative to its own pulse 1; bar = mean over the muscles, whisker = SD, dots = the muscles, thin
grey lines join the same muscle across conditions.

`PAPER` below sets **the intensity per muscle and per protocol** — one entry per muscle, or
`"default"` for all of them. Set `SAVE` to write a 300 dpi PNG.


In [ ]:

# ======== paper figure settings ========
PAPER_POLARITY = "anodic"          # "anodic" or "cathodic"
PAPER_MUSCLES  = ["Flex. digitorum (R)", "Flex. carpi rad. (R)", "Ext. digitorum (L)"]

PAPER_USE_MT = True                # True  = every muscle at its own motor threshold (MT above),
                                   #         except MT_PIN, which keeps its PAPER_MA value
                                   # False = PAPER_MA for everyone, the old behaviour
PAPER_MA = {   # intensity per protocol, per muscle ("default" covers the rest)
    "burst": {"Flex. digitorum (R)": 35, "Flex. carpi rad. (R)": 35, "Ext. digitorum (L)": 35,
              "default": 35},
    "arcex": {"Flex. digitorum (R)": 110, "Flex. carpi rad. (R)": 105, "Ext. digitorum (L)": 110,
              "default": 110},
}

def paper_amp(mode):
    """The mA of each muscle for this protocol: its own threshold measured BEFORE lidocaine
    (so before and with-lidocaine are compared at the same intensity), PAPER_MA for MT_PIN."""
    if not PAPER_USE_MT:
        return PAPER_MA[mode]
    return mt_amp((mode, PAPER_POLARITY, "before"), pin=PAPER_MA[mode])

for m_ in PAPER_MUSCLES:             # what each muscle of the figure will be plotted at
    print(f"{m_:22s} 30 Hz {str(paper_amp('burst').get(m_, '-')):>5s} mA"
          f"    ARC-EX {str(paper_amp('arcex').get(m_, '-')):>5s} mA"
          + ("   (pinned)" if m_ in MT_PIN else ""))
SAVE = None                        # e.g. "figures/paper_burst_vs_arcex_anodic.png"
# =======================================

specs = [
    dict(label="30 Hz burst · before lido", csv=D + FILE[("burst", PAPER_POLARITY, "before")],
         amp=paper_amp("burst"), colour="black",   linestyle="-",  hatch=""),
    dict(label="30 Hz burst · with lido",   csv=D + FILE[("burst", PAPER_POLARITY, "lidocaine")],
         amp=paper_amp("burst"), colour="black",   linestyle="--", hatch="///"),
    dict(label="ARC-EX · before lido",      csv=D + FILE[("arcex", PAPER_POLARITY, "before")],
         amp=paper_amp("arcex"), colour="#d62728", linestyle="-",  hatch=""),
    dict(label="ARC-EX · with lido",        csv=D + FILE[("arcex", PAPER_POLARITY, "lidocaine")],
         amp=paper_amp("arcex"), colour="#d62728", linestyle="--", hatch="///"),
]
rest = fig_train_modes(specs, PAPER_MUSCLES, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS,
                       guard_ms=GUARD_MS, title=f"{PAPER_POLARITY} - 30 Hz vs ARC-EX", save=SAVE)
for k, v in rest.items():
    print(f"{k:28s} mean of pulses 2-{N_PULSES} = {np.round(v, 0)} % of pulse 1  (per muscle)")


### First look at the EMG — then choose the muscles

One panel per muscle that has a threshold in **both** polarities: the cathodic and the anodic
train at that muscle's own `MT+MT_STEPS`, with the per-pulse peak-to-peak underneath. Keep the
muscles whose response is clear and repeatable, drop the ones that are noise. The cell prints a
ready-made `DEP_MUSCLES = [...]` line — copy it into the comparison below and delete what you
do not want.


In [ ]:
# ======== settings ========
PICK_MODE = "burst"          # "burst" or "arcex" - whose trains to look at
# ==========================

sp_c, sp_a = mt_specs(PICK_MODE, STATE)
amp_pick = tuple(mt_amp((PICK_MODE, pol, STATE), pin={}) for pol in ("cathodic", "anodic"))
cand = mt_common(PICK_MODE, STATE)

compare_at_intensity([sp_c["csv"], sp_a["csv"]], None, amp=amp_pick, muscles=cand,
                     labels=("cathodic", "anodic"), normalize="none",
                     colours=[MT_COL["cathodic"], MT_COL["anodic"]], hatches=["", "///"],
                     title=f"{NAME[PICK_MODE]} \u00b7 {STATE} lidocaine \u00b7 each muscle at its {AT}"
                           " \u2014 which of these are real?", **KW)

print(f"{len(cand)} candidate muscles - copy the line below, drop the noisy ones:\n")
print("DEP_MUSCLES = [" + ", ".join(f'"{m}"' for m in cand) + "]")


## Baseline cathodic vs anodic — depression along the train (all pre-lidocaine)

The two numbers asked for, per muscle: **2nd pulse vs 1st** and **mean of pulses 2–10 vs 1st**,
as % of the first pulse (100 % = no change, below = the response drops after the first pulse).
One figure per stimulation mode; only muscles that respond in **both** polarities are used, so the
two bars are paired — the grey lines join the same muscle. Everything here is **before lidocaine**.

Every muscle is taken at **its own motor threshold**, moved `MT_STEPS` intensities up — the same
intensity the rest of the notebook analyses it at, not one mA for the whole body. The **first panel is the 1st pulse itself, in mV** — the other two are each polarity normalised
to its own 1st pulse, so they say how much the train depresses but nothing about which polarity
responds more strongly. That is what the mV panel answers.

`DEP_MUSCLES` restricts the comparison to the muscles you picked above, `metric="diff"` switches
the ratio panels to mV, and `DEP_SAVE = True` writes the figure and the per-muscle table.


In [ ]:
# ======== settings ========
DEP_MUSCLES = ["Flex. digitorum (R)",   # None = every muscle with a
               "Flex. carpi rad. (R)"]   # threshold in both polarities
DEP_METRIC  = "ratio"                         # "ratio" = % of pulse 1 | "diff" = mV difference
DEP_SAVE    = False                           # True -> figures/ and results/ files
# ==========================

tables = {}
for mode, name in (("burst", "30 Hz burst"), ("arcex", "ARC-EX")):
    specs = mt_specs(mode, STATE)             # cathodic and anodic, same colours as above
    for sp in specs:                              # ...each muscle at its own threshold (+MT_STEPS).
        sp["amp"] = mt_amp((mode, sp["label"], STATE), pin={})   # pin={} = no hand-picked mA:
                                                  # a paired group comparison needs one rule for
                                                  # every muscle, MT_PIN included
    tag = f"{mode}_{AT}_{STATE}"
    want = DEP_MUSCLES or mt_common(mode, STATE)
    use = [m for m in want if all(m in sp["amp"] for sp in specs)]
    if set(want) - set(use):        # asked for, but no threshold in both polarities here
        print(f"{name}: no threshold in both polarities for "
              + ", ".join(sorted(set(want) - set(use))))
    tables[mode] = fig_depression(
        specs, use, metric=DEP_METRIC, with_p1=True,
        title=f"{name} \u00b7 {STATE} lidocaine \u00b7 each muscle at its {AT} \u2014 cathodic vs anodic",
        csv_out=(f"results/depression_{tag}.csv" if DEP_SAVE else None),
        save=(f"figures/depression_{tag}.png" if DEP_SAVE else None), **KW)
    print()


### The same numbers as a table

`depression_stats` returns one row per condition × muscle: pulse-1, pulse-2 and mean(2–10) in mV,
their differences from pulse 1, and the two percentages.

In [ ]:
import pandas as pd
df = pd.concat({m: pd.DataFrame(t) for m, t in tables.items()}, names=["mode"]).reset_index(level=0)
pd.set_option("display.width", 160, "display.max_columns", 20)
print(df.round(3).to_string(index=False))
